# 🌿 PlantGuard AI — Complete GPU Training Pipeline (Google Colab)

This notebook trains a high-accuracy **MobileNetV2** deep learning model on the **PlantVillage Dataset (54,000+ images, 38 classes)**.

### ⚡ Quick Start:
1. Make sure T4 GPU is active: **Runtime → Change runtime type → T4 GPU**.
2. You can either run **Cell 1 (All-in-One Automated Training)** with a single click, or run the step-by-step cells below.

## 🚀 OPTION 1: ALL-IN-ONE AUTOMATED TRAINING (Recommended — 1 Click)
Run this single cell to download the dataset, setup GPU, build the model, train both phases, and download the 3 files automatically.

In [ ]:
# =========================================================================
# 🌟 ALL-IN-ONE COMPLETE TRAINING SCRIPT (RUN THIS CELL)
# =========================================================================
import os, sys, json, time, zipfile
import tensorflow as tf
from tensorflow.keras import mixed_precision
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2

print('=' * 65)
print('🌿 PLANTGUARD AI: HIGH-ACCURACY MODEL TRAINING')
print('=' * 65)

# 1. GPU Check & Mixed Precision
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'[1/6] ✅ GPU Active: {gpus[0].name}')
    mixed_precision.set_global_policy('mixed_float16')
    print('      ✅ Mixed Precision (FP16) enabled for 2x faster training')
else:
    print('[1/6] ⚠️ No GPU found! Please select Runtime -> Change runtime type -> T4 GPU')

# 2. Dataset Download & Verification
DATASET_DIR = 'dataset_raw/PlantVillage-Dataset-master/raw/color'
if not os.path.exists(DATASET_DIR):
    print('[2/6] 📥 Downloading PlantVillage dataset archive (~800MB)...')
    !wget -q --show-progress -O plantvillage.zip https://github.com/spMohanty/PlantVillage-Dataset/archive/refs/heads/master.zip
    print('      📦 Extracting dataset images...')
    with zipfile.ZipFile('plantvillage.zip', 'r') as zip_ref:
        zip_ref.extractall('dataset_raw')

if not os.path.exists(DATASET_DIR):
    # Fallback path if folder structure differs
    for root, dirs, files in os.walk('dataset_raw'):
        if len(dirs) >= 30:
            DATASET_DIR = root
            break

classes = sorted([d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d))])
num_classes = len(classes)
print(f'[2/6] ✅ Dataset verified: {num_classes} plant disease classes loaded.')

# 3. Save class_names.json
os.makedirs('model', exist_ok=True)
class_map = {i: name for i, name in enumerate(classes)}
with open('model/class_names.json', 'w') as f:
    json.dump(class_map, f, indent=2)
print('      ✅ Saved model/class_names.json')

# 4. tf.data Data Pipelines
print('[3/6] ⚙️ Building high-speed data pipelines...')
IMG_SIZE = (224, 224)
BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset='training',
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=True
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset='validation',
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=False
)

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal_and_vertical'),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomContrast(0.15)
], name='data_augmentation')

train_data = train_ds.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=AUTOTUNE)\
                     .map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)\
                     .prefetch(AUTOTUNE)

val_data = val_ds.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=AUTOTUNE)\
                 .prefetch(AUTOTUNE)

# 5. Build Model Architecture
print('[4/6] 🏗️ Assembling MobileNetV2 Transfer Learning Architecture...')
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(256, activation='relu', kernel_regularizer=l2(0.001))(x)
x = Dropout(0.4)(x)
predictions = Dense(num_classes, activation='softmax', dtype='float32')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 6. Phase 1 Training (Warmup Head)
print('\n' + '=' * 65)
print('[5/6] 🚀 PHASE 1: Training Classification Head (5 Epochs)...')
print('=' * 65)
history1 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5,
    verbose=1
)

# 7. Phase 2 Training (Fine-Tuning Top 40 Convolutional Layers)
print('\n' + '=' * 65)
print('[6/6] 🚀 PHASE 2: Deep Fine-Tuning Top 40 MobileNetV2 Layers...')
print('=' * 65)
base_model.trainable = True
for layer in base_model.layers[:-40]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-6, verbose=1),
    tf.keras.callbacks.ModelCheckpoint('model/plant_model.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
]

history2 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=callbacks,
    verbose=1
)

# 8. Final Evaluation & Save
val_loss, val_acc = model.evaluate(val_data)
print('\n' + '=' * 65)
print(f'🎉 FINAL VALIDATION ACCURACY : {val_acc * 100:.2f}%')
print(f'   FINAL VALIDATION LOSS     : {val_loss:.4f}')
print('=' * 65)

model.save('model/plant_model.keras')
model.save('model/plant_model.h5')
print('✅ Saved model/plant_model.keras')
print('✅ Saved model/plant_model.h5')

# 9. Download Files Automatically
from google.colab import files
print('\n📥 Downloading trained files to your computer...')
files.download('model/plant_model.h5')
files.download('model/plant_model.keras')
files.download('model/class_names.json')
print('🎉 All done! Move these 3 files into your local project model/ directory.')


## 🛠️ OPTION 2: STEP-BY-STEP EXECUTION
If you prefer running individual steps one by one, use the cells below in order.

In [ ]:
# Step 1: Check GPU & Mixed Precision
import tensorflow as tf
from tensorflow.keras import mixed_precision

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'✅ GPU Active: {gpus[0].name}')
    mixed_precision.set_global_policy('mixed_float16')
    print('✅ Mixed precision FP16 enabled')
else:
    print('⚠️ No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU')


In [ ]:
# Step 2: Download & Extract Dataset
import os, zipfile

DATASET_DIR = 'dataset_raw/PlantVillage-Dataset-master/raw/color'
if not os.path.exists(DATASET_DIR):
    print('Downloading dataset archive...')
    !wget -q --show-progress -O plantvillage.zip https://github.com/spMohanty/PlantVillage-Dataset/archive/refs/heads/master.zip
    print('Extracting...')
    with zipfile.ZipFile('plantvillage.zip', 'r') as zip_ref:
        zip_ref.extractall('dataset_raw')

if not os.path.exists(DATASET_DIR):
    for root, dirs, files in os.walk('dataset_raw'):
        if len(dirs) >= 30:
            DATASET_DIR = root
            break

classes = sorted([d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d))])
num_classes = len(classes)
print(f'✅ Found {num_classes} classes at: {DATASET_DIR}')


In [ ]:
# Step 3: Create tf.data Pipeline & Save class_names.json
import json
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

IMG_SIZE = (224, 224)
BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE

os.makedirs('model', exist_ok=True)
class_map = {i: name for i, name in enumerate(classes)}
with open('model/class_names.json', 'w') as f:
    json.dump(class_map, f, indent=2)
print('✅ Saved model/class_names.json')

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset='training',
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=True
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_DIR,
    validation_split=0.2,
    subset='validation',
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=False
)

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal_and_vertical'),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomContrast(0.15)
])

train_data = train_ds.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=AUTOTUNE)\
                     .map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)\
                     .prefetch(AUTOTUNE)

val_data = val_ds.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=AUTOTUNE)\
                 .prefetch(AUTOTUNE)
print('✅ tf.data pipelines ready!')


In [ ]:
# Step 4: Build MobileNetV2 Model (Ensures 'model' is defined!)
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2

num_classes = len(classes) if 'classes' in globals() else 38

base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(256, activation='relu', kernel_regularizer=l2(0.001))(x)
x = Dropout(0.4)(x)
predictions = Dense(num_classes, activation='softmax', dtype='float32')(x)

model = Model(inputs=base_model.input, outputs=predictions)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print('✅ Model assembled and compiled successfully!')


In [ ]:
# Step 5: Phase 1 — Warmup Head Training
print('🚀 Starting Phase 1 Training (Warmup Head on T4 GPU)...')
history1 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5,
    verbose=1
)


In [ ]:
# Step 6: Phase 2 — Deep Fine-Tuning Top 40 Convolutional Layers
print('🚀 Unfreezing top 40 MobileNetV2 layers for deep fine-tuning...')
base_model.trainable = True
for layer in base_model.layers[:-40]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-6, verbose=1),
    tf.keras.callbacks.ModelCheckpoint('model/plant_model.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
]

history2 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=callbacks,
    verbose=1
)


In [ ]:
# Step 7: Final Evaluation & Download
from google.colab import files

val_loss, val_acc = model.evaluate(val_data)
print(f'🎉 FINAL ACCURACY: {val_acc * 100:.2f}% | LOSS: {val_loss:.4f}')

model.save('model/plant_model.keras')
model.save('model/plant_model.h5')

print('📥 Downloading model files...')
files.download('model/plant_model.h5')
files.download('model/plant_model.keras')
files.download('model/class_names.json')
print('✅ Download complete!')
